# 再現可能なデータ分割（Repeatable splitting）

**学習目標**
 * トレーニング / 検証 / テストの分割を作成するさまざまな方法の影響を調べる

## 概要
    
機械学習において再現性（repeatability）は重要です。今実行した処理と 5 分後に実行した処理で異なる結果が得られると、実験が困難になります。言い換えると、加えた変更によって改善が得られたのかどうかを判断することが難しくなります。

In [1]:
from google.cloud import bigquery

<h3> シンプルな機械学習モデルの作成 </h3>

使用するデータセットは、航空会社の到着データに関する <a href="https://bigquery.cloud.google.com/table/bigquery-samples:airline_ontime_data.flights">BigQuery の公開データセット</a> です。リンクをクリックして、列名を確認してください。[Details]（詳細）タブに切り替えてレコード数が 7,000 万件であることを確認し、[Preview]（プレビュー）タブに切り替えていくつかの行を確認します。
<p>
出発遅延（departure delay）に基づいて航空会社の到着遅延（arrival delay）を予測したいとします。使用するモデルはゼロバイアスの線形モデルです:
$$ delay_{arrival} = lpha * delay_{departure} $$
<p>
モデルをトレーニングすることは、$\alpha$ の適切な値を推定することです。 
<p>
alpha を推定する 1 つのアプローチは、次の式を使用することです:
$$ \alpha = \frac{\sum delay_{departure} delay_{arrival} }{  \sum delay_{departure}^2 } $$
ニューヨーク発ロサンゼルス行きの便と、オースティン発インディアナポリス行きの便（飛行距離が短く、空港の混雑度が低い）とではこの関係が異なるという考え方を捉えたいため、空港ペアごとに異なる $\alpha$ を計算します。簡単にするために、このモデルはデンバーとロサンゼルス間のフライトにのみ適用します。

<h2> 単純なランダム分割（再現性なし） </h2>

In [2]:
compute_alpha = """
#standardSQL
SELECT
    SAFE_DIVIDE(
    SUM(arrival_delay * departure_delay),
    SUM(departure_delay * departure_delay)) AS alpha
FROM
(
    SELECT
        RAND() AS splitfield,
        arrival_delay,
        departure_delay
    FROM
        `bigquery-samples.airline_ontime_data.flights`
    WHERE
        departure_airport = 'DEN'
        AND arrival_airport = 'LAX'
)
WHERE
    splitfield < 0.8
"""

In [3]:
results = bigquery.Client().query(compute_alpha).to_dataframe()
alpha = results["alpha"][0]
print(alpha)

0.9765937005026405


<h3> 次のようにトレーニング データとテスト データで RMSE を計算することの何が問題でしょうか？ </h3>

In [4]:
compute_rmse = """
#standardSQL
SELECT
    dataset,
    SQRT(
        AVG(
            (arrival_delay - ALPHA * departure_delay) *
            (arrival_delay - ALPHA * departure_delay)
        )
    ) AS rmse,
    COUNT(arrival_delay) AS num_flights
FROM (
    SELECT
        IF (RAND() < 0.8, 'train', 'eval') AS dataset,
        arrival_delay,
        departure_delay
    FROM
        `bigquery-samples.airline_ontime_data.flights`
    WHERE
        departure_airport = 'DEN'
        AND arrival_airport = 'LAX' )
GROUP BY
    dataset
"""
bigquery.Client().query(
    compute_rmse.replace("ALPHA", str(alpha))
).to_dataframe()

,dataset,rmse,num_flights
0,train,13.088724,63765
1,eval,13.064110,15924


ヒント:
* compute_rmse クエリでは、compute_alpha クエリとまったく同じトレーニング データを本当に取得できていますか？
* compute_alpha ブロックと compute_rmse ブロックを再実行するたびに、同じ結果が得られますか？

<h3> 正しくトレーニングと評価を行うにはどうすればよいでしょうか？ </h3>
<br/>
実際のトレーニング データとホールドアウト（評価）データを使用して RMSE を計算する正しい方法は次のとおりです。これがどれほど難しく感じられるかに注目してください。

計算自体は正しくなりましたが、実験にはまだ再現性がありません。

数回実行してみてください。毎回同じ結果が得られますか？

In [5]:
train_and_eval_rand = """
#standardSQL
WITH
    alldata AS (
        SELECT
            IF (RAND() < 0.8, 'train', 'eval') AS dataset,
            arrival_delay,
            departure_delay
        FROM
            `bigquery-samples.airline_ontime_data.flights`
        WHERE
        departure_airport = 'DEN'
        AND arrival_airport = 'LAX' ),
    training AS (
        SELECT
            SAFE_DIVIDE(
                SUM(arrival_delay * departure_delay),
                SUM(departure_delay * departure_delay)) AS alpha
        FROM
            alldata
        WHERE
            dataset = 'train' )

SELECT
    MAX(alpha) AS alpha,
    dataset,
    SQRT(
        AVG(
            (arrival_delay - alpha * departure_delay) *
            (arrival_delay - alpha * departure_delay)
        )
    ) AS rmse,
    COUNT(arrival_delay) AS num_flights
FROM
    alldata,
    training
GROUP BY
    dataset
"""

In [6]:
bigquery.Client().query(train_and_eval_rand).to_dataframe()

,alpha,dataset,rmse,num_flights
0,0.974766,train,13.030279,63642
1,0.974766,eval,13.294002,16047


<h2> 日付のハッシュ（HASH）を使用したデータの分割 </h2>

日付ごとに分割してトレーニングしましょう。

In [7]:
compute_alpha = """
#standardSQL
SELECT
    SAFE_DIVIDE(
        SUM(arrival_delay * departure_delay),
        SUM(departure_delay * departure_delay)) AS alpha
FROM
    `bigquery-samples.airline_ontime_data.flights`
WHERE
    departure_airport = 'DEN'
    AND arrival_airport = 'LAX'
    AND ABS(MOD(FARM_FINGERPRINT(date), 10)) < 8
"""
results = bigquery.Client().query(compute_alpha).to_dataframe()
alpha = results["alpha"][0]
print(alpha)

0.9758039143620403


これで alpha を使用して RMSE を計算できるようになりました。alpha 値には再現性があるため、compute_rmse 内の alpha が compute_alpha で計算された alpha と異なることを心配する必要はありません。

In [8]:
compute_rmse = """
#standardSQL
SELECT
    IF(ABS(MOD(FARM_FINGERPRINT(date), 10)) < 8, 'train', 'eval') AS dataset,
    SQRT(
        AVG(
            (arrival_delay - ALPHA * departure_delay) *
            (arrival_delay - ALPHA * departure_delay)
        )
    ) AS rmse,
    COUNT(arrival_delay) AS num_flights
FROM
    `bigquery-samples.airline_ontime_data.flights`
WHERE
    departure_airport = 'DEN'
    AND arrival_airport = 'LAX'
GROUP BY
    dataset
"""
print(
    bigquery.Client()
    .query(compute_rmse.replace("ALPHA", str(alpha)))
    .to_dataframe()
    .head()
)

  dataset       rmse  num_flights
0    eval  12.764685        15671
1   train  13.160712        64018


また、分割を正しく行うと、評価データセットの RMSE がトレーニング データセットの RMSE からより大きく離れることにも注目してください。これは想定されることです。RAND() のケースでは、同じ日のフライト間に高い相関関係があるため、トレーニング データセットと評価データセットの間でリーク（情報漏洩）が発生していました。
<p>
これは、機械学習の分割を誤った方法で行うことの最大の危険性の 1 つです -- <b>自分のモデルがどれほど優れているかについて、誤った自信を抱くことになります！</b>

Copyright 2020 Google Inc.
Licensed under the Apache License, Version 2.0 (the "License"); you may not use this file except in compliance with the License. You may obtain a copy of the License at
http://www.apache.org/licenses/LICENSE-2.0
Unless required by applicable law or agreed to in writing, software distributed under the License is distributed on an "AS IS" BASIS, WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied. See the License for the specific language governing permissions and limitations under the License.